In [12]:
import os 
import warnings

import cv2 
import h5py
import numpy as np
import torch 
import torch.nn as nn
import torchvision.transforms as T 
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large
from tqdm import tqdm

In [13]:
def get_model(model_path, device=None):
    checkpoints = torch.load(model_path, map_location=device, weights_only=True)
    model = deeplabv3_mobilenet_v3_large(num_classes=2, aux_loss=True).to(device)
    model.load_state_dict(checkpoints, strict=False)
    return model 

def image_preprocess_transforms(mean=(0.4611, 0.4359, 0.3905), std=(0.2193, 0.2150, 0.2109)):
    common_transforms = T.Compose([T.ToTensor(), T.Normalize(mean, std),])
    return common_transforms

def compute_segmentation_mask(model, img_tens):
    with torch.inference_mode():
        out = model(img_tens)["out"]
        out_mask = torch.argmax(out, dim=1, keepdim=True).permute(0, 2, 3, 1)[0].cpu().numpy().squeeze().astype(np.int32)
    return out_mask
    
def calculate_contours(mask, half):
    mask_padded = np.pad(mask * 255, pad_width=((half, half), (half, half)), mode='constant')
    
    canny = cv2.Canny(mask_padded.astype(np.uint8), 225, 255)
    canny = cv2.dilate(canny, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    contours, _ = cv2.findContours(canny, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    
    if not contours:
        return None

    largest_contour = max(contours, key=cv2.contourArea)
    return largest_contour 

def output_contours_for_img(model: nn.Module,
                            img: np.ndarray,
                            img_path: str, 
                            image_size: int = 384, 
                            preprocess_transforms=image_preprocess_transforms(), 
                            device: torch.device = torch.device("cpu")):
    if not isinstance(img, np.ndarray):
        raise TypeError(f"img should be a np.ndarray, got {type(img)}")
    
    # pad image for segmentation to work
    # need smaller image (since that is what seg model was trained on)
    half = image_size // 2
    imH, imW, C = img.shape
    resized_img = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_NEAREST)
    scale_x = imW / image_size
    scale_y = imH / image_size
    
    # convert image to tensor and normalize
    img_tens = preprocess_transforms(resized_img)
    img_tens = torch.unsqueeze(img_tens, dim=0) # type: ignore 
    
    img_tens = img_tens.to(device)
    
    mask = compute_segmentation_mask(model, img_tens)
    
    contour = calculate_contours(mask, half)
    
    if contour is None: 
        warnings.warn(f"No contour found for image {img_path}")
        return None
    
    return {"contour": contour.astype(np.int16),
            "half": half, 
            "scale_x": scale_x, 
            "scale_y": scale_y}
    
def save_contours_to_hdf5(contour_dict, filepath='contours.h5'):
    """
    Save all contours to a single HDF5 file
    """
    with h5py.File(filepath, 'w') as f:
        for img_id, data in tqdm(contour_dict.items(), desc="Saving"):
            if data is None:  # Skip failed segmentations
                continue
            
            # Create group for this image
            grp = f.create_group(img_id)
            
            # Save contour array
            grp.create_dataset('contour', data=data['contour'], 
                                compression='gzip', dtype='int16')
            
            # Save metadata as attributes
            grp.attrs['scale_x'] = data['scale_x']
            grp.attrs['scale_y'] = data['scale_y'] 
            grp.attrs['half'] = data['half']    

def precompute_contours_pipeline(model_path, img_paths, device, output_path='contours.h5'): 
    model = get_model(model_path, device=device)
    model.eval()
    
    contour_set = {}
    failed_images = []
    
    for path in tqdm(img_paths, desc="Processing images"):
        try:
            # extract image id 
            img_id = os.path.splitext(os.path.basename(path))[0]
            
            img = cv2.imread(path)
            if img is None:
                print(f"Failed to load: {path}")
                failed_images.append(path)
                continue
                
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            contour_item = output_contours_for_img(
                model=model, 
                img=img, 
                img_path=path, 
                device=device
            )
            
            contour_set[img_id] = contour_item  # Fixed: use img_id not "img_id"
            
        except Exception as e:
            print(f"Error processing {path}: {e}")
            failed_images.append(path)
    
    save_contours_to_hdf5(contour_set, filepath=output_path)
    
    print(f"Completed. Processed: {len(contour_set)}, Failed: {len(failed_images)}")
    if failed_images:
        print(f"Failed images: {failed_images[:10]}")  # Show first 10
    
    return contour_set, failed_images

In [14]:
class ECGSegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, img_paths, image_size=384, transform=None):
        self.img_paths = img_paths
        self.image_size = image_size
        self.transform = transform or image_preprocess_transforms()
        
    def __len__(self):
        return len(self.img_paths)
    
    def __getitem__(self, idx):
        path = self.img_paths[idx]
        img_id = os.path.splitext(os.path.basename(path))[0]
        
        try:
            img = cv2.imread(path)
            if img is None:
                return None, None, None
                
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            imH, imW = img.shape[:2]
            
            resized = cv2.resize(img, (self.image_size, self.image_size), 
                               interpolation=cv2.INTER_NEAREST)
            tensor = self.transform(resized)
            
            metadata = {
                'img_id': img_id,
                'scale_x': imW / self.image_size,
                'scale_y': imH / self.image_size,
                'path': path
            }
            
            return tensor, metadata, True
            
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return None, {'img_id': img_id, 'path': path}, False

def collate_fn(batch):
    """Custom collate to handle failed images"""
    tensors, metadata, success = zip(*batch)
    
    valid_tensors = [t for t, s in zip(tensors, success) if s]
    valid_metadata = [m for m, s in zip(metadata, success) if s]
    failed_metadata = [m for m, s in zip(metadata, success) if not s]
    
    if valid_tensors:
        batch_tensor = torch.stack(valid_tensors)
        return batch_tensor, valid_metadata, failed_metadata
    else:
        return None, [], failed_metadata

In [15]:
def precompute_with_dataloader(model_path, img_paths, device, 
                              output_path='contours.h5',
                              batch_size=8, num_workers=4):
    model = get_model(model_path, device=device)
    model.eval()
    
    dataset = ECGSegmentationDataset(img_paths)
    dataloader = torch.utils.data.DataLoader(
        dataset, 
        batch_size=batch_size,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True  # Faster GPU transfer
    )
    
    contour_set = {}
    failed_images = []
    image_size = 384
    half = image_size // 2
    
    for batch_tensor, valid_metadata, failed_metadata in tqdm(dataloader, 
                                                              desc="Processing"):
        # Track failed images
        failed_images.extend([m['path'] for m in failed_metadata])
        
        if batch_tensor is None:
            continue
            
        # Process batch through model
        batch_tensor = batch_tensor.to(device)
        with torch.inference_mode():
            masks = model(batch_tensor)["out"]
            masks = torch.argmax(masks, dim=1).cpu().numpy()
        
        # Extract contours
        for mask, metadata in zip(masks, valid_metadata):
            contour = calculate_contours(mask, half)
            if contour is not None:
                contour_set[metadata['img_id']] = {
                    'contour': contour.astype(np.int16),
                    'half': half,
                    'scale_x': metadata['scale_x'],
                    'scale_y': metadata['scale_y']
                }
    
    save_contours_to_hdf5(contour_set, filepath=output_path)
    return contour_set, failed_images

In [16]:
with open("/kaggle/input/bhf-reference-files/broken_images_list.txt", "r") as file:
    broken_train_images = [line.strip() for line in file]
with open("/kaggle/input/bhf-reference-files/broken_test_images_list.txt", "r") as file:
    broken_test_images = [line.strip() for line in file]
with open("/kaggle/input/bhf-reference-files/valid_images_list.txt", "r") as file:
    valid_train_images = [line.strip() for line in file]
with open("/kaggle/input/bhf-reference-files/valid_test_images_list.txt", "r") as file:
    valid_test_images = [line.strip() for line in file]

In [17]:
device = torch.device(
            "mps" if torch.backends.mps.is_available() and torch.backends.mps.is_built() \
            else ("cuda" if torch.cuda.is_available() 
            else "cpu")
      )
print(f"Using device = {device}")

Using device = cuda


In [18]:
model_path = "/kaggle/input/document-detection/pytorch/default/1/model_mbv3_iou_mix_2C049.pth"

In [19]:
img_paths = valid_train_images[:1000]

In [20]:
precompute_with_dataloader(model_path=model_path, 
                           img_paths = img_paths, 
                           device = device,
                           output_path = "/kaggle/working/output.h5")

Saving: 100%|██████████| 1000/1000 [00:00<00:00, 1407.30it/s]


({'train_011565': {'contour': array([[[278, 267]],
   
          [[277, 268]],
   
          [[276, 268]],
   
          ...,
   
          [[281, 267]],
   
          [[280, 267]],
   
          [[279, 267]]], dtype=int16),
   'half': 192,
   'scale_x': 9.010416666666666,
   'scale_y': 4.947916666666667},
  'train_011207': {'contour': array([[[385, 258]],
   
          [[384, 259]],
   
          [[383, 259]],
   
          ...,
   
          [[388, 258]],
   
          [[387, 258]],
   
          [[386, 258]]], dtype=int16),
   'half': 192,
   'scale_x': 9.010416666666666,
   'scale_y': 4.947916666666667},
  'train_011641': {'contour': array([[[287, 257]],
   
          [[286, 258]],
   
          [[285, 258]],
   
          ...,
   
          [[290, 257]],
   
          [[289, 257]],
   
          [[288, 257]]], dtype=int16),
   'half': 192,
   'scale_x': 9.010416666666666,
   'scale_y': 4.947916666666667},
  'train_011441': {'contour': array([[[278, 234]],
   
          [[277, 235]